# TalkSense — Simulador de Event Hub (Fabric Notebook)

**Objetivo**: este notebook substitui o *voice agent* real e o **Azure Event Hubs** durante uma **Prova de Conceito (POC)**, gerando eventos sintéticos no mesmo formato JSON documentado em [`infra/docs/event-hub-message-format.md`](../infra/docs/event-hub-message-format.md) e enviando-os continuamente para o **Fabric Eventstream** (via endpoint *Custom App*, compatível com Event Hub) ou diretamente para a **Eventhouse** (ingestão KQL).

Arquitetura simulada:

```
[ESTE NOTEBOOK]  ──JSON──▶  Fabric Eventstream (Custom App)  ──▶  Eventhouse (KQL DB)  ──▶  Power BI
      ▲
      └── substitui o Voice Agent + Azure Event Hubs reais nesta POC
```

> ⚠️ **Somente para POC/teste.** Não use dados reais de clientes. Não deixe rodando indefinidamente contra um ambiente de produção — monitore custos de ingestão/Eventstream.

## Pré-requisitos

1. Um **Fabric Eventstream** criado, com uma fonte **"Custom App"** adicionada (isso expõe uma connection string compatível com Event Hub) — ver [`infra/docs/fabric-configuration.md`](../infra/docs/fabric-configuration.md).
2. Ou, alternativamente, uma **Eventhouse (KQL Database)** já provisionada, com a tabela `RawTelemetry` criada.
3. Este notebook deve ser importado para um **Workspace do Microsoft Fabric** e anexado a um Lakehouse/Environment (não é necessário Lakehouse para este notebook, apenas o kernel Python/PySpark padrão do Fabric).

In [ ]:
# Instala os SDKs necessários no ambiente do notebook Fabric
%pip install --quiet azure-eventhub azure-identity azure-kusto-data azure-kusto-ingest

In [ ]:
# ============================================================================
# Configuração do simulador — ajuste antes de executar
# ============================================================================

# --- Destino dos eventos ---
# "eventstream": envia via SDK do Event Hubs para o Azure Event Hub (Managed Identity), que alimenta o Fabric Eventstream (RECOMENDADO)
# "eventhouse" : ingestão direta na Eventhouse via Kusto Queued Ingestion (bypassa o Eventstream)
SINK_MODE = "eventstream"  # "eventstream" | "eventhouse"

# --- Conexão Event Hub (Managed Identity / Entra ID — SEM keys/connection string) ---
# Use o FQDN do namespace (output 'eventHubNamespaceFqdn' do Bicep / 'eventhub_namespace_fqdn' do Terraform).
EVENTHUB_FULLY_QUALIFIED_NAMESPACE = "<SEU_NAMESPACE>.servicebus.windows.net"
EVENTHUB_NAME = "evh-voiceagent-telemetry"  # nome do Event Hub
# A identidade que executa este notebook precisa da role RBAC 'Azure Event Hubs Data Sender' no Event Hub.

# --- Conexão Eventhouse (Kusto) — usada apenas se SINK_MODE == "eventhouse" ---
KUSTO_CLUSTER_URI = "https://<seu-cluster>.kusto.fabric.microsoft.com"
KUSTO_DATABASE = "VoiceAgentDB"
KUSTO_TABLE_RAW = "RawTelemetry"  # tabela que recebe o JSON bruto (ver fabric-configuration.md)

# --- Parâmetros da simulação ---
CONVERSATIONS_PER_MINUTE = 6        # taxa de novas conversas simuladas
MIN_TURNS, MAX_TURNS = 3, 10         # turnos por conversa
TURN_PACE_SECONDS = (0.5, 2.5)       # pausa entre turnos (acelerado p/ POC; em produção seria ~10-30s)
MAX_RUNTIME_MINUTES = 30             # None = roda indefinidamente até ser interrompido manualmente (célula > Interromper)
RANDOM_SEED = None                   # defina um int para reprodutibilidade

In [ ]:
# ============================================================================
# Geradores de eventos sintéticos (mesmo schema de infra/docs/event-hub-message-format.md)
# ============================================================================
import random
import uuid
import json
import time
import hashlib
from datetime import datetime, timezone

if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)

CHANNELS = ["phone", "chat", "whatsapp"]
SEGMENTS = ["Premium", "Standard", "Basic"]
INTENTS = ["saldo", "transferencia", "investimentos", "sinistro", "seguros", "cartao", "emprestimo", "cancelamento", "duvida_geral"]

OUTCOMES_WEIGHTS = [
    ("resolved_by_ai", 0.70),
    ("transferred_to_agent", 0.15),
    ("customer_hangup", 0.10),
    ("system_timeout", 0.05),
]

# (eventName, eventCategory, peso)
EVENT_POOL_WEIGHTS = [
    ("no_comprehension", "error", 0.04),
    ("knowledge_gap", "error", 0.04),
    ("system_error", "error", 0.02),
    ("returned_to_ura", "navigation", 0.03),
    ("agent_requested", "handoff", 0.05),
    ("barge_in", "interaction", 0.06),
    ("silence_timeout", "interaction", 0.02),
]


def _now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def _anon_id():
    return hashlib.sha256(str(uuid.uuid4()).encode()).hexdigest()[:32]


def _weighted_choice(pairs):
    items = [p[0] for p in pairs]
    weights = [p[-1] for p in pairs]
    return random.choices(items, weights=weights, k=1)[0]


def make_conversation_started(conversation_id):
    return {
        "eventType": "conversation_started",
        "timestamp": _now_iso(),
        "conversationId": conversation_id,
        "payload": {
            "sessionId": f"sess-{uuid.uuid4().hex[:12]}",
            "channel": random.choice(CHANNELS),
            "customerIdAnon": _anon_id(),
            "customerSegment": random.choice(SEGMENTS),
            "initialIntent": random.choice(INTENTS),
            "originQueue": random.choice(INTENTS),
            "metadata": {"region": "BR-SP", "languageCode": "pt-BR", "source": "eventhub-simulator"},
        },
    }


def make_turn_completed(conversation_id, turn_index, intent):
    latency = random.randint(300, 4500)
    return {
        "eventType": "turn_completed",
        "timestamp": _now_iso(),
        "conversationId": conversation_id,
        "payload": {
            "turnIndex": turn_index,
            "userUtterance": "[MASKED]",
            "agentResponse": f"Resposta simulada para intent '{intent}'.",
            "intent": intent,
            "intentConfidence": round(random.uniform(0.55, 0.99), 2),
            "sentiment": {
                "overall": random.choice(["positive", "neutral", "negative"]),
                "score": round(random.uniform(-1, 1), 2),
            },
            "latencyMs": latency,
            "ttsEnabled": random.random() > 0.1,
            "bargeInDetected": random.random() < 0.1,
        },
    }


def make_conversation_event(conversation_id, turn_index, event_name, event_category, details=None):
    return {
        "eventType": "conversation_event",
        "timestamp": _now_iso(),
        "conversationId": conversation_id,
        "payload": {
            "eventName": event_name,
            "eventCategory": event_category,
            "turnIndex": turn_index,
            "details": details or {},
        },
    }


def make_csat_received(conversation_id):
    score = random.choices([5, 4, 3, 2, 1], weights=[0.45, 0.25, 0.15, 0.10, 0.05], k=1)[0]
    return {
        "eventType": "csat_received",
        "timestamp": _now_iso(),
        "conversationId": conversation_id,
        "payload": {
            "score": score,
            "scale": "1-5",
            "comment": None,
            "collectedVia": random.choice(["post_call_ivr", "sms", "email"]),
        },
    }

In [ ]:
# ============================================================================
# Clientes de envio (sinks) — Eventstream (Event Hub SDK) ou Eventhouse (Kusto Ingest)
# ============================================================================
from azure.eventhub import EventHubProducerClient, EventData
from azure.identity import DefaultAzureCredential

_eh_producer = None


def get_eventhub_producer():
    global _eh_producer
    if _eh_producer is None:
        # Autenticação keyless via Managed Identity (ou az login / VS Code / variáveis de ambiente) — sem SAS keys.
        _eh_producer = EventHubProducerClient(
            fully_qualified_namespace=EVENTHUB_FULLY_QUALIFIED_NAMESPACE,
            eventhub_name=EVENTHUB_NAME,
            credential=DefaultAzureCredential(),
        )
    return _eh_producer


def send_via_eventstream(events):
    producer = get_eventhub_producer()
    batch = producer.create_batch()
    for evt in events:
        data = EventData(json.dumps(evt))
        data.properties = {"conversationId": evt["conversationId"]}
        try:
            batch.add(data)
        except ValueError:
            producer.send_batch(batch)
            batch = producer.create_batch()
            batch.add(data)
    if len(batch) > 0:
        producer.send_batch(batch)


_kusto_ingest_client = None


def get_kusto_ingest_client():
    global _kusto_ingest_client
    if _kusto_ingest_client is None:
        from azure.kusto.ingest import QueuedIngestClient
        from azure.kusto.data import KustoConnectionStringBuilder

        kcsb = KustoConnectionStringBuilder.with_az_cli_authentication(KUSTO_CLUSTER_URI)
        _kusto_ingest_client = QueuedIngestClient(kcsb)
    return _kusto_ingest_client


def send_via_eventhouse(events):
    from azure.kusto.ingest import IngestionProperties, DataFormat
    import io

    client = get_kusto_ingest_client()
    props = IngestionProperties(database=KUSTO_DATABASE, table=KUSTO_TABLE_RAW, data_format=DataFormat.MULTIJSON)
    ndjson = "\n".join(json.dumps(e) for e in events)
    stream = io.StringIO(ndjson)
    client.ingest_from_stream(stream, ingestion_properties=props)


def send_events(events):
    if not events:
        return
    if SINK_MODE == "eventstream":
        send_via_eventstream(events)
    elif SINK_MODE == "eventhouse":
        send_via_eventhouse(events)
    else:
        raise ValueError(f"SINK_MODE invalido: {SINK_MODE}")

In [ ]:
# ============================================================================
# Loop principal — gera e envia conversas continuamente (Interrompa a célula para parar)
# ============================================================================

def simulate_conversation():
    conversation_id = f"conv-{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}-{uuid.uuid4().hex[:6]}"

    send_events([make_conversation_started(conversation_id)])

    n_turns = random.randint(MIN_TURNS, MAX_TURNS)
    current_intent = random.choice(INTENTS)
    special_event_turn = random.randint(1, n_turns) if random.random() < 0.35 else None

    for turn_index in range(1, n_turns + 1):
        time.sleep(random.uniform(*TURN_PACE_SECONDS))
        turn_events = [make_turn_completed(conversation_id, turn_index, current_intent)]
        if turn_index == special_event_turn:
            event_name, event_category, _weight = random.choices(
                EVENT_POOL_WEIGHTS, weights=[w for *_, w in EVENT_POOL_WEIGHTS], k=1
            )[0]
            turn_events.append(make_conversation_event(conversation_id, turn_index, event_name, event_category))
        send_events(turn_events)

    outcome = _weighted_choice(OUTCOMES_WEIGHTS)
    closing_events = [
        make_conversation_event(
            conversation_id,
            n_turns,
            "conversation_ended",
            "completion",
            details={"outcome": outcome, "durationSeconds": n_turns * 8, "totalTurns": n_turns},
        )
    ]
    if random.random() < 0.6:
        closing_events.append(make_csat_received(conversation_id))
    send_events(closing_events)

    return conversation_id, n_turns, outcome


print(f"Iniciando simulador de Event Hub | sink={SINK_MODE} | taxa={CONVERSATIONS_PER_MINUTE} conversas/min")
start_time = time.time()
conversation_count = 0

try:
    while True:
        if MAX_RUNTIME_MINUTES is not None and (time.time() - start_time) > MAX_RUNTIME_MINUTES * 60:
            print(f"Tempo maximo de execucao ({MAX_RUNTIME_MINUTES} min) atingido. Encerrando.")
            break

        conv_id, n_turns, outcome = simulate_conversation()
        conversation_count += 1
        elapsed_min = (time.time() - start_time) / 60
        print(f"[{conversation_count:04d}] {conv_id} | {n_turns} turnos | outcome={outcome} | {elapsed_min:.1f} min decorridos")

        avg_pace = sum(TURN_PACE_SECONDS) / 2
        pause_between_conversations = max(0.0, (60 / CONVERSATIONS_PER_MINUTE) - (n_turns * avg_pace))
        time.sleep(pause_between_conversations)

except KeyboardInterrupt:
    print(f"Interrompido manualmente apos {conversation_count} conversas simuladas.")

print(f"Total de conversas enviadas: {conversation_count}")

## Verificando os dados na Eventhouse

Após alguns minutos de execução, valide a chegada dos dados executando esta consulta KQL no **KQL Queryset** da Eventhouse:

```kql
RawTelemetry
| where ingestion_time() > ago(10m)
| summarize count() by eventType = tostring(parse_json(RawJson).eventType)
```

(ajuste o nome da coluna/tabela conforme o schema definido em [`infra/docs/fabric-configuration.md`](../infra/docs/fabric-configuration.md)).

## Parar a simulação

- Interrompa a célula do loop principal (ícone de "Stop" / `Interromper` na barra do notebook Fabric), ou
- Defina `MAX_RUNTIME_MINUTES` para um valor finito antes de executar.

## Próximos passos

1. Confirme que os eventos aparecem no [`powerbi/TalkSense.pbip`](../powerbi/README.md) (troque a fonte de dados para Direct Lake/DirectQuery apontando para a Eventhouse real).
2. Ajuste `CONVERSATIONS_PER_MINUTE` e `TURN_PACE_SECONDS` para simular picos de carga.
3. Para um teste de carga maior, execute múltiplas instâncias deste notebook em paralelo (Fabric permite agendar/rodar notebooks via pipeline).